In [1]:
!pip install panns-inference -q
!pip install librosa scikit-learn pandas -q

import warnings
warnings.filterwarnings('ignore')

# Create comprehensive directory structure
import os
from pathlib import Path
from datetime import datetime

# Base directories
BASE_DIRS = {
    'outputs': '/kaggle/working/outputs',
    'models': '/kaggle/working/models',
    'results': '/kaggle/working/results',
    'logs': '/kaggle/working/logs',
    'embeddings': '/kaggle/working/embeddings',
    'plots': '/kaggle/working/plots',
    'checkpoints': '/kaggle/working/checkpoints'
}

for name, path in BASE_DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✅ Created: {path}")

# Create timestamped run directory
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'/kaggle/working/run_{RUN_TIMESTAMP}'
os.makedirs(RUN_DIR, exist_ok=True)

# Create subdirectories in run folder
for subdir in ['logs', 'results', 'embeddings', 'plots']:
    os.makedirs(os.path.join(RUN_DIR, subdir), exist_ok=True)

print(f"\n🚀 Run directory: {RUN_DIR}")
print(f"📅 Timestamp: {RUN_TIMESTAMP}")
print("\n✅ All directories created successfully!")


✅ Created: /kaggle/working/outputs
✅ Created: /kaggle/working/models
✅ Created: /kaggle/working/results
✅ Created: /kaggle/working/logs
✅ Created: /kaggle/working/embeddings
✅ Created: /kaggle/working/plots
✅ Created: /kaggle/working/checkpoints

🚀 Run directory: /kaggle/working/run_20260120_063740
📅 Timestamp: 20260120_063740

✅ All directories created successfully!


In [2]:
import os
import sys
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors
from dataclasses import dataclass, field
from typing import List, Tuple, Dict
import json
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time
import traceback
import pickle

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Setup comprehensive logging
import logging

class ExperimentLogger:
    """Comprehensive logging system for the entire experiment"""
    
    def __init__(self, run_dir, run_timestamp):
        self.run_dir = run_dir
        self.run_timestamp = run_timestamp
        self.log_file = os.path.join(run_dir, 'logs', 'experiment.log')
        self.metrics_file = os.path.join(run_dir, 'logs', 'metrics.json')
        self.progress_file = os.path.join(run_dir, 'logs', 'progress.json')
        
        # Setup file logger
        self.logger = logging.getLogger('DCASE_Experiment')
        self.logger.setLevel(logging.INFO)
        
        # File handler
        fh = logging.FileHandler(self.log_file)
        fh.setLevel(logging.INFO)
        
        # Console handler
        ch = logging.StreamHandler(sys.stdout)
        ch.setLevel(logging.INFO)
        
        # Formatter
        formatter = logging.Formatter(
            '%(asctime)s | %(levelname)s | %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )
        fh.setFormatter(formatter)
        ch.setFormatter(formatter)
        
        self.logger.addHandler(fh)
        self.logger.addHandler(ch)
        
        # Initialize metrics storage
        self.metrics = {
            'run_info': {
                'timestamp': run_timestamp,
                'run_dir': run_dir,
                'start_time': datetime.now().isoformat()
            },
            'machines': {},
            'summary': {}
        }
        
        # Initialize progress tracking
        self.progress = {
            'completed_machines': [],
            'failed_machines': [],
            'current_machine': None,
            'total_machines': 0,
            'start_time': time.time()
        }
        
        self.log("=" * 80)
        self.log("🚀 DCASE 2025 Anomaly Detection - PANNs + 4-Layer Strategy")
        self.log("=" * 80)
        self.log(f"Run ID: {run_timestamp}")
        self.log(f"Run Directory: {run_dir}")
        self.log(f"Log File: {self.log_file}")
    
    def log(self, message, level='info'):
        """Log message to both file and console"""
        if level == 'info':
            self.logger.info(message)
        elif level == 'warning':
            self.logger.warning(message)
        elif level == 'error':
            self.logger.error(message)
    
    def save_metrics(self):
        """Save metrics to JSON file"""
        with open(self.metrics_file, 'w') as f:
            json.dump(self.metrics, f, indent=2)
        self.log(f"💾 Metrics saved to: {self.metrics_file}")
    
    def save_progress(self):
        """Save progress to JSON file"""
        self.progress['elapsed_time_minutes'] = (time.time() - self.progress['start_time']) / 60
        with open(self.progress_file, 'w') as f:
            json.dump(self.progress, f, indent=2)
    
    def update_machine_start(self, machine_id):
        """Log machine processing start"""
        self.progress['current_machine'] = machine_id
        self.save_progress()
        self.log(f"\n{'='*80}")
        self.log(f"📊 Processing Machine: {machine_id}")
        self.log(f"{'='*80}")
    
    def update_machine_complete(self, machine_id, results):
        """Log machine processing completion"""
        self.progress['completed_machines'].append(machine_id)
        self.progress['current_machine'] = None
        self.metrics['machines'][machine_id] = results
        self.save_progress()
        self.save_metrics()
        self.log(f"✅ Completed: {machine_id}")
    
    def update_machine_failed(self, machine_id, error):
        """Log machine processing failure"""
        self.progress['failed_machines'].append({
            'machine': machine_id,
            'error': str(error),
            'timestamp': datetime.now().isoformat()
        })
        self.progress['current_machine'] = None
        self.save_progress()
        self.log(f"❌ Failed: {machine_id} - Error: {error}", level='error')
    
    def finalize(self, summary):
        """Finalize experiment and save final summary"""
        self.metrics['summary'] = summary
        self.metrics['run_info']['end_time'] = datetime.now().isoformat()
        self.metrics['run_info']['total_time_minutes'] = summary.get('total_time_minutes', 0)
        self.save_metrics()
        
        self.log("\n" + "=" * 80)
        self.log("✅ EXPERIMENT COMPLETED")
        self.log("=" * 80)
        self.log(f"Total machines processed: {len(self.progress['completed_machines'])}")
        self.log(f"Failed machines: {len(self.progress['failed_machines'])}")
        self.log(f"Total time: {summary.get('total_time_minutes', 0):.1f} minutes")
        self.log(f"Average AUC: {summary.get('average_auc', 0):.2f}%")

# Initialize logger
logger = ExperimentLogger(RUN_DIR, RUN_TIMESTAMP)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.log(f"🖥️  Device: {device}")
logger.log(f"🔧 PyTorch version: {torch.__version__}")
logger.log(f"🔧 CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    logger.log(f"🔧 GPU: {torch.cuda.get_device_name(0)}")
    logger.log(f"🔧 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

logger.log("\n✅ Libraries imported and logger initialized successfully!")


2026-01-20 06:37:49 | INFO | ================================================================================
2026-01-20 06:37:49 | INFO | 🚀 DCASE 2025 Anomaly Detection - PANNs + 4-Layer Strategy
2026-01-20 06:37:49 | INFO | ================================================================================
2026-01-20 06:37:49 | INFO | Run ID: 20260120_063740
2026-01-20 06:37:49 | INFO | Run Directory: /kaggle/working/run_20260120_063740
2026-01-20 06:37:49 | INFO | Log File: /kaggle/working/run_20260120_063740/logs/experiment.log
2026-01-20 06:37:49 | INFO | 🖥️  Device: cuda
2026-01-20 06:37:49 | INFO | 🔧 PyTorch version: 2.8.0+cu126
2026-01-20 06:37:49 | INFO | 🔧 CUDA available: True
2026-01-20 06:37:49 | INFO | 🔧 GPU: Tesla T4
2026-01-20 06:37:49 | INFO | 🔧 GPU Memory: 15.83 GB
2026-01-20 06:37:49 | INFO | 
✅ Libraries imported and logger initialized successfully!


In [3]:
@dataclass
class Config:
    """Enhanced configuration with supplemental data and attributes support"""
    
    # ========== PATHS ==========
    BASE_PATH: str = '/kaggle/input/dcase-dataset-2025/development/development'
    OUTPUT_DIR: str = RUN_DIR
    RESULTS_DIR: str = os.path.join(RUN_DIR, 'results')
    EMBEDDINGS_DIR: str = os.path.join(RUN_DIR, 'embeddings')
    PLOTS_DIR: str = os.path.join(RUN_DIR, 'plots')
    LOGS_DIR: str = os.path.join(RUN_DIR, 'logs')
    
    # ========== MACHINES (ALL 7) ==========
    MACHINES: List[str] = field(default_factory=lambda: [
        'dev_fan',
        'dev_gearbox',
        'dev_bearing',
        'dev_slider',
        'dev_ToyCar',
        'dev_ToyTrain',
        'dev_valve'
    ])
    
    MACHINE_NAMES: dict = field(default_factory=lambda: {
        'dev_fan': 'fan',
        'dev_gearbox': 'gearbox',
        'dev_bearing': 'bearing',
        'dev_slider': 'slider',
        'dev_ToyCar': 'ToyCar',
        'dev_ToyTrain': 'ToyTrain',
        'dev_valve': 'valve'
    })
    
    # ========== AUDIO PROCESSING ==========
    SAMPLE_RATE: int = 16000
    AUDIO_LEN: int = 160000
    
    # Multi-resolution spectrogram (YOUR CONTRIBUTION!)
    STFT_SIZES: List[int] = field(default_factory=lambda: [256, 1024, 4096])
    SPEC_CHANNELS: int = 3
    SPEC_HEIGHT: int = 128
    SPEC_WIDTH: int = 320
    
    # Inverse-Mel settings
    USE_INVERSE_MEL: bool = True
    FMIN: int = 500
    FMAX: int = 4000
    
    # ========== SUPPLEMENTAL DATA (NEW!) ==========
    USE_SUPPLEMENTAL: bool = True  # Use supplemental folder for extra training data
    SUPPLEMENTAL_WEIGHT: float = 0.5  # Weight for supplemental samples (vs train samples)
    MAX_SUPPLEMENTAL_SAMPLES: int = 500  # Limit supplemental samples per machine
    
    # ========== ATTRIBUTES (NEW!) ==========
    USE_ATTRIBUTES: bool = True  # Use attributes.csv for metadata
    USE_SECTION_NORMALIZATION: bool = True  # Section-aware normalization
    USE_DOMAIN_WEIGHTING: bool = True  # Weight samples by domain
    
    # ========== MODEL ==========
    EMBEDDING_DIM: int = 2048  # PANNs CNN14 output dimension
    
    # ========== IMPROVEMENT LAYERS (YOUR CONTRIBUTION!) ==========
    USE_SCORE_NORM: bool = True
    K_VALUES: List[int] = field(default_factory=lambda: [1, 3, 5, 7, 10])
    K_WEIGHTS: List[float] = field(default_factory=lambda: [0.1, 0.2, 0.4, 0.2, 0.1])
    USE_TTA: bool = False  # Test-Time Augmentation (disabled for speed)
    USE_MAHALANOBIS: bool = True
    MAHALANOBIS_WEIGHT: float = 0.4
    
    # ========== EVALUATION ==========
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 2
    
    # ========== SAVING OPTIONS ==========
    SAVE_EMBEDDINGS: bool = True  # Save extracted embeddings
    SAVE_PLOTS: bool = True  # Save visualization plots
    SAVE_DETAILED_RESULTS: bool = True  # Save per-sample predictions
    
    def __post_init__(self):
        """Validate configuration"""
        # Ensure directories exist
        for dir_path in [self.OUTPUT_DIR, self.RESULTS_DIR, self.EMBEDDINGS_DIR, 
                        self.PLOTS_DIR, self.LOGS_DIR]:
            os.makedirs(dir_path, exist_ok=True)

# Initialize configuration
config = Config()

# Log configuration
logger.log("\n📋 Configuration:")
logger.log(f"  Base Path: {config.BASE_PATH}")
logger.log(f"  Output Directory: {config.OUTPUT_DIR}")
logger.log(f"  Number of Machines: {len(config.MACHINES)}")
logger.log(f"  Machines: {', '.join(config.MACHINES)}")

logger.log("\n🎵 Audio Processing:")
logger.log(f"  Sample Rate: {config.SAMPLE_RATE} Hz")
logger.log(f"  Audio Length: {config.AUDIO_LEN} samples ({config.AUDIO_LEN/config.SAMPLE_RATE} sec)")
logger.log(f"  STFT Sizes: {config.STFT_SIZES}")
logger.log(f"  Spectrogram Shape: ({config.SPEC_CHANNELS}, {config.SPEC_HEIGHT}, {config.SPEC_WIDTH})")
logger.log(f"  Frequency Range: {config.FMIN}-{config.FMAX} Hz")

logger.log("\n🆕 Enhanced Features:")
logger.log(f"  Use Supplemental Data: {config.USE_SUPPLEMENTAL}")
if config.USE_SUPPLEMENTAL:
    logger.log(f"    - Max Supplemental Samples: {config.MAX_SUPPLEMENTAL_SAMPLES}")
    logger.log(f"    - Supplemental Weight: {config.SUPPLEMENTAL_WEIGHT}")
logger.log(f"  Use Attributes Metadata: {config.USE_ATTRIBUTES}")
if config.USE_ATTRIBUTES:
    logger.log(f"    - Section Normalization: {config.USE_SECTION_NORMALIZATION}")
    logger.log(f"    - Domain Weighting: {config.USE_DOMAIN_WEIGHTING}")

logger.log("\n🧠 Model & Detection:")
logger.log(f"  Backbone: PANNs CNN14 (pre-trained)")
logger.log(f"  Embedding Dimension: {config.EMBEDDING_DIM}")
logger.log(f"  k-NN k values: {config.K_VALUES}")
logger.log(f"  k-NN weights: {config.K_WEIGHTS}")
logger.log(f"  Mahalanobis enabled: {config.USE_MAHALANOBIS}")
logger.log(f"  Score normalization: {config.USE_SCORE_NORM}")

logger.log("\n💾 Saving Options:")
logger.log(f"  Save Embeddings: {config.SAVE_EMBEDDINGS}")
logger.log(f"  Save Plots: {config.SAVE_PLOTS}")
logger.log(f"  Save Detailed Results: {config.SAVE_DETAILED_RESULTS}")

# Save configuration to file
config_dict = {
    'base_path': config.BASE_PATH,
    'machines': config.MACHINES,
    'audio': {
        'sample_rate': config.SAMPLE_RATE,
        'audio_len': config.AUDIO_LEN,
        'stft_sizes': config.STFT_SIZES,
        'fmin': config.FMIN,
        'fmax': config.FMAX
    },
    'supplemental': {
        'use_supplemental': config.USE_SUPPLEMENTAL,
        'max_samples': config.MAX_SUPPLEMENTAL_SAMPLES,
        'weight': config.SUPPLEMENTAL_WEIGHT
    },
    'attributes': {
        'use_attributes': config.USE_ATTRIBUTES,
        'section_normalization': config.USE_SECTION_NORMALIZATION,
        'domain_weighting': config.USE_DOMAIN_WEIGHTING
    },
    'model': {
        'embedding_dim': config.EMBEDDING_DIM,
        'k_values': config.K_VALUES,
        'k_weights': config.K_WEIGHTS,
        'mahalanobis_weight': config.MAHALANOBIS_WEIGHT
    }
}

config_file = os.path.join(config.LOGS_DIR, 'config.json')
with open(config_file, 'w') as f:
    json.dump(config_dict, f, indent=2)

logger.log(f"\n💾 Configuration saved to: {config_file}")
logger.log("\n✅ Configuration loaded successfully!")


2026-01-20 06:37:49 | INFO | 
📋 Configuration:
2026-01-20 06:37:49 | INFO |   Base Path: /kaggle/input/dcase-dataset-2025/development/development
2026-01-20 06:37:49 | INFO |   Output Directory: /kaggle/working/run_20260120_063740
2026-01-20 06:37:49 | INFO |   Number of Machines: 7
2026-01-20 06:37:49 | INFO |   Machines: dev_fan, dev_gearbox, dev_bearing, dev_slider, dev_ToyCar, dev_ToyTrain, dev_valve
2026-01-20 06:37:49 | INFO | 
🎵 Audio Processing:
2026-01-20 06:37:49 | INFO |   Sample Rate: 16000 Hz
2026-01-20 06:37:49 | INFO |   Audio Length: 160000 samples (10.0 sec)
2026-01-20 06:37:49 | INFO |   STFT Sizes: [256, 1024, 4096]
2026-01-20 06:37:49 | INFO |   Spectrogram Shape: (3, 128, 320)
2026-01-20 06:37:49 | INFO |   Frequency Range: 500-4000 Hz
2026-01-20 06:37:49 | INFO | 
🆕 Enhanced Features:
2026-01-20 06:37:49 | INFO |   Use Supplemental Data: True
2026-01-20 06:37:49 | INFO |     - Max Supplemental Samples: 500
2026-01-20 06:37:49 | INFO |     - Supplemental Weight: 0.

In [4]:
# # ============================================================
# # FIX LIBROSA/NUMBA COMPATIBILITY
# # ============================================================

# print("🔧 Fixing librosa/numba compatibility issue...")

# # Uninstall and reinstall with compatible versions
# import sys
# !{sys.executable} -m pip uninstall -y numba librosa -q
# !{sys.executable} -m pip install numba==0.56.4 -q
# !{sys.executable} -m pip install librosa==0.10.0 -q

# print("✅ Reinstalled compatible versions")
# print("⚠️  IMPORTANT: Please restart the kernel and re-run from Cell 1")
# print("   (Runtime -> Restart Runtime)")


In [5]:
# ============================================================
# INVERSE-MEL UTILITIES (WITHOUT LOGGER)
# ============================================================

def hz_to_mel(hz):
    """Convert Hz to Mel scale"""
    return 2595 * np.log10(1 + hz / 700)

def mel_to_hz(mel):
    """Convert Mel to Hz scale"""
    return 700 * (10**(mel / 2595) - 1)

def create_inverse_mel_filterbank(sr, n_fft, n_mels, fmin, fmax):
    """
    Create inverse mel filterbank for reconstruction
    This achieves higher frequency resolution in industrial sound range (500-4000 Hz)
    """
    mel_min = hz_to_mel(fmin)
    mel_max = hz_to_mel(fmax)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    
    fbank = np.zeros((n_mels, n_fft // 2 + 1))
    
    for i in range(1, n_mels + 1):
        left, center, right = bin_points[i-1], bin_points[i], bin_points[i+1]
        for j in range(left, center):
            if center != left:
                fbank[i-1, j] = (j - left) / (center - left)
        for j in range(center, right):
            if right != center:
                fbank[i-1, j] = (right - j) / (right - center)
    
    inverse_fbank = np.linalg.pinv(fbank)
    return fbank, inverse_fbank

def extract_multi_resolution_spectrogram(audio, sr=16000, stft_sizes=[256, 1024, 4096],
                                        n_mels=128, fmin=500, fmax=4000, target_width=320):
    """
    Extract multi-resolution spectrogram (YOUR INNOVATION!)
    
    Captures complementary time-frequency information:
    - Fine (256): Transient events, high-frequency details
    - Mid (1024): Rhythmic patterns, mechanical vibrations  
    - Coarse (4096): Overall harmonic structure, low-frequency content
    """
    spectrograms = []
    
    for n_fft in stft_sizes:
        hop_length = n_fft // 4
        
        # Create mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, fmin=fmin, fmax=fmax
        )
        
        # Apply inverse-Mel transformation for better frequency resolution
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_power = librosa.db_to_power(mel_db)
        
        _, inv_bank = create_inverse_mel_filterbank(sr, n_fft, n_mels, fmin, fmax)
        inv_spec = inv_bank @ mel_power
        inv_spec_db = librosa.power_to_db(inv_spec, ref=np.max)
        inv_spec_db = inv_spec_db[:n_mels, :]
        
        # Resize to fixed width
        if inv_spec_db.shape[1] < target_width:
            pad_width = target_width - inv_spec_db.shape[1]
            inv_spec_db = np.pad(inv_spec_db, ((0, 0), (0, pad_width)), mode='constant')
        else:
            inv_spec_db = inv_spec_db[:, :target_width]
        
        spectrograms.append(inv_spec_db)
    
    # Stack into 3-channel image
    multi_res_spec = np.stack(spectrograms, axis=0)  # (3, 128, 320)
    
    # Normalize each channel independently
    for i in range(len(stft_sizes)):
        mean = multi_res_spec[i].mean()
        std = multi_res_spec[i].std()
        multi_res_spec[i] = (multi_res_spec[i] - mean) / (std + 1e-8)
    
    return multi_res_spec

# Test spectrogram extraction
print("🧪 Testing multi-resolution spectrogram extraction...")
test_audio = np.random.randn(160000)  # 10 seconds @ 16kHz
test_spec = extract_multi_resolution_spectrogram(
    test_audio, 
    sr=config.SAMPLE_RATE,
    stft_sizes=config.STFT_SIZES,
    n_mels=config.SPEC_HEIGHT,
    fmin=config.FMIN,
    fmax=config.FMAX,
    target_width=config.SPEC_WIDTH
)

print(f"✅ Test spectrogram shape: {test_spec.shape}")
print(f"   Expected: ({config.SPEC_CHANNELS}, {config.SPEC_HEIGHT}, {config.SPEC_WIDTH})")
print(f"   Channel 0 (Fine) range: [{test_spec[0].min():.3f}, {test_spec[0].max():.3f}]")
print(f"   Channel 1 (Mid) range: [{test_spec[1].min():.3f}, {test_spec[1].max():.3f}]")
print(f"   Channel 2 (Coarse) range: [{test_spec[2].min():.3f}, {test_spec[2].max():.3f}]")

assert test_spec.shape == (config.SPEC_CHANNELS, config.SPEC_HEIGHT, config.SPEC_WIDTH), \
    f"Spectrogram shape mismatch! Got {test_spec.shape}"

print("✅ Inverse-Mel utilities loaded and tested successfully!")


🧪 Testing multi-resolution spectrogram extraction...
✅ Test spectrogram shape: (3, 128, 320)
   Expected: (3, 128, 320)
   Channel 0 (Fine) range: [-0.756, 1.793]
   Channel 1 (Mid) range: [-1.663, 1.016]
   Channel 2 (Coarse) range: [-1.019, 0.981]
✅ Inverse-Mel utilities loaded and tested successfully!


In [6]:
class DcaseDataset(Dataset):
    """DCASE anomaly detection dataset with multi-resolution spectrograms"""
    
    def __init__(self, file_paths, labels, domains, sections, config):
        self.file_paths = file_paths
        self.labels = labels
        self.domains = domains
        self.sections = sections
        self.config = config
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        # Load audio
        audio, _ = librosa.load(self.file_paths[idx], sr=self.config.SAMPLE_RATE, mono=True)
        
        # Pad or trim to fixed length
        if len(audio) < self.config.AUDIO_LEN:
            audio = np.pad(audio, (0, self.config.AUDIO_LEN - len(audio)))
        else:
            audio = audio[:self.config.AUDIO_LEN]
        
        # Extract multi-resolution spectrogram
        spectrogram = extract_multi_resolution_spectrogram(
            audio, sr=self.config.SAMPLE_RATE,
            stft_sizes=self.config.STFT_SIZES,
            n_mels=self.config.SPEC_HEIGHT,
            fmin=self.config.FMIN,
            fmax=self.config.FMAX,
            target_width=self.config.SPEC_WIDTH
        )
        
        return {
            'audio': torch.FloatTensor(audio),
            'spectrogram': torch.FloatTensor(spectrogram),
            'label': self.labels[idx],
            'domain': self.domains[idx],
            'section': self.sections[idx]
        }

def parse_filename(filename):
    """Parse filename to extract metadata"""
    basename = os.path.basename(filename)
    
    # Extract section (e.g., section_00, section_01)
    section = basename.split('_')[1] if 'section' in basename else '00'
    
    # Extract domain (source=0, target=1)
    domain = 0 if 'source' in basename else 1
    
    # Extract label (normal=0, anomaly=1)
    label = 0 if 'normal' in basename else 1
    
    return section, domain, label

def load_attributes(attributes_file):
    """Load and parse attributes.csv file"""
    if not os.path.exists(attributes_file):
        logger.log(f"⚠️  Attributes file not found: {attributes_file}", level='warning')
        return None
    
    try:
        df = pd.read_csv(attributes_file)
        logger.log(f"✅ Loaded attributes with {len(df)} rows and columns: {list(df.columns)}")
        return df
    except Exception as e:
        logger.log(f"⚠️  Error loading attributes: {e}", level='warning')
        return None

def load_machine_data(machine_dir, machine_name, config):
    """
    Load training, supplemental, and test data for a specific machine
    
    Returns:
        train_data: dict with files, labels, domains, sections, weights
        test_data: dict with files, labels, domains, sections
        attributes_df: pandas DataFrame with metadata (or None)
    """
    machine_path = os.path.join(machine_dir, machine_name)
    train_dir = os.path.join(machine_path, 'train')
    test_dir = os.path.join(machine_path, 'test')
    supplemental_dir = os.path.join(machine_path, 'supplemental')
    attributes_file = os.path.join(machine_path, 'attributes_00.csv')
    
    logger.log(f"\n📂 Loading data for {machine_name}...")
    
    # Load attributes
    attributes_df = None
    if config.USE_ATTRIBUTES:
        attributes_df = load_attributes(attributes_file)
    
    # === TRAINING DATA ===
    train_files = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith('.wav')])
    train_labels = []
    train_domains = []
    train_sections = []
    train_weights = []
    
    for f in train_files:
        section, domain, label = parse_filename(f)
        train_labels.append(label)
        train_domains.append(domain)
        train_sections.append(section)
        train_weights.append(1.0)  # Full weight for training data
    
    logger.log(f"  📁 Train: {len(train_files)} files")
    logger.log(f"     - Source: {sum(np.array(train_domains)==0)}")
    logger.log(f"     - Target: {sum(np.array(train_domains)==1)}")
    logger.log(f"     - Sections: {len(set(train_sections))} unique")
    
    # === SUPPLEMENTAL DATA (NEW!) ===
    supplemental_files = []
    supplemental_labels = []
    supplemental_domains = []
    supplemental_sections = []
    supplemental_weights = []
    
    if config.USE_SUPPLEMENTAL and os.path.exists(supplemental_dir):
        all_supplemental = sorted([os.path.join(supplemental_dir, f) 
                                  for f in os.listdir(supplemental_dir) if f.endswith('.wav')])
        
        # Limit supplemental samples
        if len(all_supplemental) > config.MAX_SUPPLEMENTAL_SAMPLES:
            # Sample randomly but keep source/target balance
            source_supp = [f for f in all_supplemental if 'source' in f]
            target_supp = [f for f in all_supplemental if 'target' in f]
            
            n_source = min(len(source_supp), config.MAX_SUPPLEMENTAL_SAMPLES // 2)
            n_target = min(len(target_supp), config.MAX_SUPPLEMENTAL_SAMPLES // 2)
            
            np.random.seed(42)
            selected_source = list(np.random.choice(source_supp, n_source, replace=False))
            selected_target = list(np.random.choice(target_supp, n_target, replace=False))
            
            supplemental_files = selected_source + selected_target
        else:
            supplemental_files = all_supplemental
        
        for f in supplemental_files:
            section, domain, label = parse_filename(f)
            supplemental_labels.append(0)  # Supplemental assumed normal
            supplemental_domains.append(domain)
            supplemental_sections.append(section)
            supplemental_weights.append(config.SUPPLEMENTAL_WEIGHT)  # Reduced weight
        
        logger.log(f"  📁 Supplemental: {len(supplemental_files)} files (max: {config.MAX_SUPPLEMENTAL_SAMPLES})")
        logger.log(f"     - Source: {sum(np.array(supplemental_domains)==0)}")
        logger.log(f"     - Target: {sum(np.array(supplemental_domains)==1)}")
        logger.log(f"     - Weight: {config.SUPPLEMENTAL_WEIGHT}")
    else:
        logger.log(f"  📁 Supplemental: Not used (enabled={config.USE_SUPPLEMENTAL}, exists={os.path.exists(supplemental_dir)})")
    
    # === COMBINE TRAINING + SUPPLEMENTAL ===
    all_train_files = train_files + supplemental_files
    all_train_labels = train_labels + supplemental_labels
    all_train_domains = train_domains + supplemental_domains
    all_train_sections = train_sections + supplemental_sections
    all_train_weights = train_weights + supplemental_weights
    
    logger.log(f"  📊 Total Training: {len(all_train_files)} files (train + supplemental)")
    
    # === TEST DATA ===
    test_files = sorted([os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.wav')])
    test_labels = []
    test_domains = []
    test_sections = []
    
    for f in test_files:
        section, domain, label = parse_filename(f)
        test_labels.append(label)
        test_domains.append(domain)
        test_sections.append(section)
    
    logger.log(f"  📁 Test: {len(test_files)} files")
    logger.log(f"     - Normal: {sum(np.array(test_labels)==0)}")
    logger.log(f"     - Anomaly: {sum(np.array(test_labels)==1)}")
    logger.log(f"     - Source: {sum(np.array(test_domains)==0)}")
    logger.log(f"     - Target: {sum(np.array(test_domains)==1)}")
    
    # Package data
    train_data = {
        'files': all_train_files,
        'labels': np.array(all_train_labels),
        'domains': np.array(all_train_domains),
        'sections': np.array(all_train_sections),
        'weights': np.array(all_train_weights)
    }
    
    test_data = {
        'files': test_files,
        'labels': np.array(test_labels),
        'domains': np.array(test_domains),
        'sections': np.array(test_sections)
    }
    
    return train_data, test_data, attributes_df

logger.log("\n✅ Enhanced dataset class loaded successfully!")


2026-01-20 06:38:03 | INFO | 
✅ Enhanced dataset class loaded successfully!


In [7]:
class PANNsFeatureExtractor:
    """
    Pre-trained PANNs CNN14 for feature extraction
    No training needed - leverages AudioSet pre-training!
    """
    
    def __init__(self, device='cuda'):
        logger.log("\n🔧 Initializing PANNs CNN14 model...")
        self.device = device
        
        try:
            from panns_inference import AudioTagging
            self.model = AudioTagging(checkpoint_path=None, device=device)
            logger.log("✅ PANNs CNN14 loaded successfully!")
            logger.log(f"   Model device: {device}")
            logger.log(f"   Pre-trained on AudioSet (2M audio clips)")
            logger.log(f"   Output embedding dimension: 2048")
        except Exception as e:
            logger.log(f"❌ Failed to load PANNs: {e}", level='error')
            raise
    
    def extract_features(self, dataloader, split_name='train'):
        """
        Extract 2048D embeddings using PANNs
        
        Args:
            dataloader: PyTorch DataLoader
            split_name: Name of split (for logging)
        
        Returns:
            embeddings: (N, 2048) numpy array
            labels: (N,) numpy array
            domains: (N,) numpy array
            sections: (N,) numpy array
        """
        embeddings_list = []
        labels_list = []
        domains_list = []
        sections_list = []
        
        self.model.model.eval()
        
        logger.log(f"\n🎵 Extracting features for {split_name} split...")
        start_time = time.time()
        
        with torch.no_grad():
            pbar = tqdm(dataloader, desc=f"Extracting {split_name} features")
            for batch_idx, batch in enumerate(pbar):
                # Use raw audio for PANNs
                audio = batch['audio'].numpy()
                
                batch_embeddings = []
                for i in range(len(audio)):
                    # PANNs expects audio as (1, samples)
                    audio_input = audio[i:i+1]
                    
                    try:
                        # Get embedding from PANNs (clipwise_output, embedding)
                        _, embedding = self.model.inference(audio_input)
                        batch_embeddings.append(embedding[0])  # Shape: (2048,)
                    except Exception as e:
                        logger.log(f"⚠️  Error extracting embedding for sample {batch_idx*len(audio)+i}: {e}", 
                                 level='warning')
                        # Use zero embedding as fallback
                        batch_embeddings.append(np.zeros(2048))
                
                embeddings_list.extend(batch_embeddings)
                labels_list.append(batch['label'].numpy())
                domains_list.append(batch['domain'].numpy())
                sections_list.append(batch['section'])
                
                # Update progress bar with stats
                if batch_idx % 10 == 0:
                    elapsed = time.time() - start_time
                    rate = (batch_idx + 1) * len(audio) / elapsed if elapsed > 0 else 0
                    pbar.set_postfix({
                        'samples': f"{len(embeddings_list)}/{len(dataloader.dataset)}",
                        'rate': f"{rate:.1f} samples/s"
                    })
        
        embeddings = np.vstack(embeddings_list)
        labels = np.concatenate(labels_list)
        domains = np.concatenate(domains_list)
        sections = np.concatenate(sections_list)
        
        elapsed_time = time.time() - start_time
        
        logger.log(f"✅ Feature extraction complete for {split_name}")
        logger.log(f"   Embeddings shape: {embeddings.shape}")
        logger.log(f"   Time: {elapsed_time:.1f}s ({len(embeddings)/elapsed_time:.1f} samples/s)")
        logger.log(f"   Mean embedding norm: {np.linalg.norm(embeddings, axis=1).mean():.3f}")
        logger.log(f"   Std embedding norm: {np.linalg.norm(embeddings, axis=1).std():.3f}")
        
        return embeddings, labels, domains, sections

logger.log("✅ PANNs feature extractor loaded successfully!")


2026-01-20 06:38:04 | INFO | ✅ PANNs feature extractor loaded successfully!


In [8]:
class FourLayerAnomalyDetector:
    """
    Enhanced 4-layer anomaly scoring strategy (YOUR INNOVATION!)
    Now with section-aware normalization and domain weighting
    """
    
    def __init__(self, config):
        self.config = config
        self.train_embeddings = None
        self.train_domains = None
        self.train_sections = None
        self.train_weights = None
        self.baseline_stats = {}
        self.section_stats = {}
    
    def fit(self, train_embeddings, train_domains, train_sections, train_weights=None):
        """Fit on training embeddings with weights and sections"""
        self.train_embeddings = train_embeddings
        self.train_domains = train_domains
        self.train_sections = train_sections
        self.train_weights = train_weights if train_weights is not None else np.ones(len(train_embeddings))
        
        logger.log(f"\n🎯 Fitting anomaly detector...")
        logger.log(f"   Training samples: {len(train_embeddings)}")
        logger.log(f"   Embedding dimension: {train_embeddings.shape[1]}")
        logger.log(f"   Unique sections: {len(np.unique(train_sections))}")
        
        # Compute baseline statistics for Layer 4 (domain-aware)
        baseline_scores = self._multi_k_ensemble(train_embeddings, train_embeddings, 
                                                 train_weights, train_weights)
        
        for domain in [0, 1]:
            domain_mask = train_domains == domain
            if domain_mask.sum() > 0:
                self.baseline_stats[domain] = {
                    'mean': baseline_scores[domain_mask].mean(),
                    'std': baseline_scores[domain_mask].std(),
                    'count': domain_mask.sum()
                }
                logger.log(f"   Domain {domain} baseline: μ={self.baseline_stats[domain]['mean']:.4f}, "
                         f"σ={self.baseline_stats[domain]['std']:.4f}, n={self.baseline_stats[domain]['count']}")
        
        # Compute section-specific statistics (NEW!)
        if self.config.USE_SECTION_NORMALIZATION:
            unique_sections = np.unique(train_sections)
            for section in unique_sections:
                section_mask = train_sections == section
                if section_mask.sum() > 5:  # Need enough samples
                    self.section_stats[section] = {
                        'mean': baseline_scores[section_mask].mean(),
                        'std': baseline_scores[section_mask].std(),
                        'count': section_mask.sum()
                    }
            logger.log(f"   Section-specific stats computed for {len(self.section_stats)} sections")
    
    def _multi_k_ensemble(self, query_embeddings, ref_embeddings, 
                         query_weights=None, ref_weights=None):
        """Layer 1: Multi-k Ensemble with optional weighting"""
        scores_all_k = []
        
        # Apply weights to reference embeddings if provided
        if ref_weights is not None and self.config.USE_DOMAIN_WEIGHTING:
            # Duplicate samples based on weights for k-NN
            weighted_ref = []
            for i, w in enumerate(ref_weights):
                # Convert weight to integer repetitions (0.5 -> skip 50% of time)
                if w >= 0.5 or np.random.rand() < w:
                    weighted_ref.append(ref_embeddings[i])
            
            if len(weighted_ref) > 0:
                ref_embeddings_weighted = np.array(weighted_ref)
            else:
                ref_embeddings_weighted = ref_embeddings
        else:
            ref_embeddings_weighted = ref_embeddings
        
        for k in self.config.K_VALUES:
            # Ensure k doesn't exceed number of samples
            k_actual = min(k, len(ref_embeddings_weighted))
            
            knn = NearestNeighbors(n_neighbors=k_actual, metric='euclidean', n_jobs=-1)
            knn.fit(ref_embeddings_weighted)
            distances, _ = knn.kneighbors(query_embeddings)
            scores_k = distances.mean(axis=1)
            scores_all_k.append(scores_k)
        
        # Weighted average
        scores_all_k = np.array(scores_all_k)
        weights = np.array(self.config.K_WEIGHTS[:len(scores_all_k)])
        weights = weights / weights.sum()  # Normalize
        ensemble_score = (scores_all_k.T @ weights).flatten()
        
        return ensemble_score
    
    def _mahalanobis_distance(self, query_embeddings, train_embeddings, train_domains):
        """Layer 2: Mahalanobis Distance per domain"""
        scores = []
        
        for domain in [0, 1]:
            domain_mask = train_domains == domain
            if domain_mask.sum() < 10:  # Need enough samples for covariance
                continue
            
            domain_embeddings = train_embeddings[domain_mask]
            mean = domain_embeddings.mean(axis=0)
            
            # Compute covariance with regularization
            cov = np.cov(domain_embeddings.T)
            
            # Add stronger regularization for stability
            reg_factor = 0.01 * np.trace(cov) / cov.shape[0]
            cov_reg = cov + reg_factor * np.eye(cov.shape[0])
            
            try:
                cov_inv = np.linalg.inv(cov_reg)
                diff = query_embeddings - mean
                mahal_dist = np.sqrt(np.sum(diff @ cov_inv * diff, axis=1))
                scores.append(mahal_dist)
            except np.linalg.LinAlgError:
                logger.log(f"⚠️  Mahalanobis computation failed for domain {domain}", level='warning')
                continue
        
        if len(scores) > 0:
            return np.min(scores, axis=0)
        else:
            return np.zeros(len(query_embeddings))
    
    def _score_fusion(self, euclidean_scores, mahalanobis_scores):
        """Layer 3: Score Fusion"""
        # Min-max normalization
        euclidean_norm = (euclidean_scores - euclidean_scores.min()) / \
                        (euclidean_scores.max() - euclidean_scores.min() + 1e-8)
        mahalanobis_norm = (mahalanobis_scores - mahalanobis_scores.min()) / \
                          (mahalanobis_scores.max() - mahalanobis_scores.min() + 1e-8)
        
        # Weighted combination
        combined = (1 - self.config.MAHALANOBIS_WEIGHT) * euclidean_norm + \
                   self.config.MAHALANOBIS_WEIGHT * mahalanobis_norm
        return combined
    
    def _domain_aware_normalization(self, scores, test_domains):
        """Layer 4a: Domain-Aware Score Normalization"""
        if not self.config.USE_SCORE_NORM:
            return scores
        
        normalized_scores = np.zeros_like(scores)
        
        for domain in [0, 1]:
            if domain not in self.baseline_stats:
                normalized_scores[test_domains == domain] = scores[test_domains == domain]
                continue
            
            domain_mask = test_domains == domain
            mean = self.baseline_stats[domain]['mean']
            std = self.baseline_stats[domain]['std']
            
            normalized_scores[domain_mask] = (scores[domain_mask] - mean) / (std + 1e-6)
        
        return normalized_scores
    
    def _section_aware_normalization(self, scores, test_sections):
        """Layer 4b: Section-Aware Normalization (NEW!)"""
        if not self.config.USE_SECTION_NORMALIZATION or len(self.section_stats) == 0:
            return scores
        
        normalized_scores = scores.copy()
        
        for section, stats in self.section_stats.items():
            section_mask = test_sections == section
            if section_mask.sum() > 0:
                mean = stats['mean']
                std = stats['std']
                normalized_scores[section_mask] = (scores[section_mask] - mean) / (std + 1e-6)
        
        return normalized_scores
    
    def predict(self, test_embeddings, test_domains, test_sections):
        """Predict anomaly scores using all 4 layers + section normalization"""
        logger.log(f"\n🔮 Computing anomaly scores...")
        logger.log(f"   Test samples: {len(test_embeddings)}")
        
        # Layer 1: Multi-k Ensemble
        start_time = time.time()
        euclidean_scores = self._multi_k_ensemble(
            test_embeddings, self.train_embeddings,
            None, self.train_weights
        )
        logger.log(f"   Layer 1 (Multi-k): {time.time()-start_time:.2f}s")
        
        # Layer 2: Mahalanobis Distance
        if self.config.USE_MAHALANOBIS:
            start_time = time.time()
            mahalanobis_scores = self._mahalanobis_distance(
                test_embeddings, self.train_embeddings, self.train_domains
            )
            logger.log(f"   Layer 2 (Mahalanobis): {time.time()-start_time:.2f}s")
            
            # Layer 3: Score Fusion
            combined_scores = self._score_fusion(euclidean_scores, mahalanobis_scores)
            logger.log(f"   Layer 3 (Fusion): Combined")
        else:
            combined_scores = euclidean_scores
        
        # Layer 4a: Domain-Aware Normalization
        start_time = time.time()
        domain_normalized = self._domain_aware_normalization(combined_scores, test_domains)
        logger.log(f"   Layer 4a (Domain norm): {time.time()-start_time:.2f}s")
        
        # Layer 4b: Section-Aware Normalization (NEW!)
        if self.config.USE_SECTION_NORMALIZATION:
            start_time = time.time()
            final_scores = self._section_aware_normalization(domain_normalized, test_sections)
            logger.log(f"   Layer 4b (Section norm): {time.time()-start_time:.2f}s")
        else:
            final_scores = domain_normalized
        
        logger.log(f"   Final score range: [{final_scores.min():.3f}, {final_scores.max():.3f}]")
        
        return final_scores

logger.log("✅ Enhanced 4-layer anomaly detector loaded successfully!")


2026-01-20 06:38:04 | INFO | ✅ Enhanced 4-layer anomaly detector loaded successfully!


In [9]:
def compute_metrics(labels, scores, domains):
    """Compute comprehensive evaluation metrics"""
    from sklearn.metrics import roc_curve, auc
    
    # Overall AUC
    auc_overall = roc_auc_score(labels, scores) * 100
    
    # Source domain AUC
    source_mask = domains == 0
    if source_mask.sum() > 0 and len(np.unique(labels[source_mask])) > 1:
        auc_source = roc_auc_score(labels[source_mask], scores[source_mask]) * 100
    else:
        auc_source = 0.0
    
    # Target domain AUC
    target_mask = domains == 1
    if target_mask.sum() > 0 and len(np.unique(labels[target_mask])) > 1:
        auc_target = roc_auc_score(labels[target_mask], scores[target_mask]) * 100
    else:
        auc_target = 0.0
    
    # Compute pAUC at FPR <= 0.1
    fpr, tpr, thresholds = roc_curve(labels, scores)
    mask = fpr <= 0.1
    if mask.sum() > 1:
        pauc = np.trapz(tpr[mask], fpr[mask]) * 10 * 100  # Scale to 0-10%
    else:
        pauc = 0.0
    
    # Official DCASE score (harmonic mean)
    if auc_overall + pauc > 0:
        official_score = 2 * auc_overall * pauc / (auc_overall + pauc)
    else:
        official_score = 0.0
    
    return {
        'auc_overall': auc_overall,
        'auc_source': auc_source,
        'auc_target': auc_target,
        'pauc': pauc,
        'official_score': official_score,
        'fpr': fpr,
        'tpr': tpr,
        'thresholds': thresholds
    }

def plot_results(metrics, machine_id, test_labels, test_scores, test_domains, config):
    """Create comprehensive visualization plots"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: ROC Curve
    ax = axes[0, 0]
    ax.plot(metrics['fpr'], metrics['tpr'], 'b-', linewidth=2, 
            label=f'AUC = {metrics["auc_overall"]:.2f}%')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.axvline(x=0.1, color='r', linestyle='--', alpha=0.5, label='FPR=0.1 (pAUC cutoff)')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC Curve - {machine_id}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Score Distribution
    ax = axes[0, 1]
    normal_scores = test_scores[test_labels == 0]
    anomaly_scores = test_scores[test_labels == 1]
    
    bins = np.linspace(test_scores.min(), test_scores.max(), 50)
    ax.hist(normal_scores, bins=bins, alpha=0.6, label=f'Normal (n={len(normal_scores)})', 
            color='green', edgecolor='black')
    ax.hist(anomaly_scores, bins=bins, alpha=0.6, label=f'Anomaly (n={len(anomaly_scores)})', 
            color='red', edgecolor='black')
    ax.set_xlabel('Anomaly Score', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'Score Distribution - {machine_id}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Domain-wise Performance
    ax = axes[1, 0]
    domains_names = ['Source', 'Target']
    aucs = [metrics['auc_source'], metrics['auc_target']]
    colors = ['#3498db', '#e74c3c']
    
    bars = ax.bar(domains_names, aucs, color=colors, edgecolor='black', linewidth=1.5)
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random (50%)')
    ax.axhline(y=metrics['auc_overall'], color='green', linestyle='--', 
               alpha=0.7, label=f'Overall ({metrics["auc_overall"]:.1f}%)')
    ax.set_ylabel('AUC (%)', fontsize=12)
    ax.set_title(f'Domain-wise Performance - {machine_id}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 100])
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # Plot 4: Metrics Summary
    ax = axes[1, 1]
    ax.axis('off')
    
    summary_text = f"""
    {'='*50}
    PERFORMANCE SUMMARY - {machine_id}
    {'='*50}
    
    Overall Metrics:
      • AUC (Overall):     {metrics['auc_overall']:>6.2f}%
      • pAUC (FPR≤0.1):    {metrics['pauc']:>6.2f}%
      • Official Score:    {metrics['official_score']:>6.2f}%
    
    Domain-specific:
      • AUC (Source):      {metrics['auc_source']:>6.2f}%
      • AUC (Target):      {metrics['auc_target']:>6.2f}%
    
    Dataset Statistics:
      • Total Samples:     {len(test_labels):>6d}
      • Normal:            {sum(test_labels==0):>6d}
      • Anomaly:           {sum(test_labels==1):>6d}
      • Source Domain:     {sum(test_domains==0):>6d}
      • Target Domain:     {sum(test_domains==1):>6d}
    
    Score Statistics:
      • Min Score:         {test_scores.min():>6.3f}
      • Max Score:         {test_scores.max():>6.3f}
      • Mean (Normal):     {normal_scores.mean():>6.3f}
      • Mean (Anomaly):    {anomaly_scores.mean():>6.3f}
    
    {'='*50}
    """
    
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    
    # Save plot
    if config.SAVE_PLOTS:
        plot_file = os.path.join(config.PLOTS_DIR, f'{machine_id}_results.png')
        plt.savefig(plot_file, dpi=150, bbox_inches='tight')
        logger.log(f"   📊 Plot saved: {plot_file}")
    
    plt.show()
    plt.close()

def save_detailed_results(machine_id, test_files, test_labels, test_scores, 
                         test_domains, test_sections, config):
    """Save per-sample detailed results"""
    if not config.SAVE_DETAILED_RESULTS:
        return
    
    # Create detailed results DataFrame
    results_df = pd.DataFrame({
        'filename': [os.path.basename(f) for f in test_files],
        'filepath': test_files,
        'label': test_labels,
        'domain': ['source' if d == 0 else 'target' for d in test_domains],
        'section': test_sections,
        'anomaly_score': test_scores,
        'prediction': (test_scores > np.median(test_scores)).astype(int)
    })
    
    # Add rank
    results_df['rank'] = results_df['anomaly_score'].rank(ascending=False).astype(int)
    
    # Sort by anomaly score (highest first)
    results_df = results_df.sort_values('anomaly_score', ascending=False)
    
    # Save to CSV
    csv_file = os.path.join(config.RESULTS_DIR, f'{machine_id}_detailed.csv')
    results_df.to_csv(csv_file, index=False)
    logger.log(f"   💾 Detailed results saved: {csv_file}")
    
    return results_df

logger.log("✅ Evaluation and visualization functions loaded successfully!")


2026-01-20 06:38:04 | INFO | ✅ Evaluation and visualization functions loaded successfully!


In [10]:
def evaluate_machine(machine_id, config, feature_extractor):
    """
    Evaluate a single machine with complete pipeline:
    1. Load data (train + supplemental + test + attributes)
    2. Extract PANNs features
    3. Apply 4-layer anomaly detection
    4. Compute metrics and visualize
    5. Save all results
    """
    
    machine_start_time = time.time()
    logger.update_machine_start(machine_id)
    
    try:
        machine_name = config.MACHINE_NAMES[machine_id]
        
        # ============================================================
        # STEP 1: LOAD DATA
        # ============================================================
        train_data, test_data, attributes_df = load_machine_data(
            config.BASE_PATH, machine_name, config
        )
        
        # ============================================================
        # STEP 2: CREATE DATASETS
        # ============================================================
        logger.log(f"\n📦 Creating datasets...")
        train_dataset = DcaseDataset(
            train_data['files'], 
            train_data['labels'],
            train_data['domains'],
            train_data['sections'],
            config
        )
        
        test_dataset = DcaseDataset(
            test_data['files'],
            test_data['labels'],
            test_data['domains'],
            test_data['sections'],
            config
        )
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config.BATCH_SIZE,
            shuffle=False, 
            num_workers=config.NUM_WORKERS,
            pin_memory=True
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=config.BATCH_SIZE,
            shuffle=False,
            num_workers=config.NUM_WORKERS,
            pin_memory=True
        )
        
        logger.log(f"   Train loader: {len(train_loader)} batches")
        logger.log(f"   Test loader: {len(test_loader)} batches")
        
        # ============================================================
        # STEP 3: EXTRACT FEATURES WITH PANNs
        # ============================================================
        extract_start = time.time()
        
        train_embeddings, train_labels, train_domains, train_sections = \
            feature_extractor.extract_features(train_loader, split_name='train')
        
        test_embeddings, test_labels, test_domains, test_sections = \
            feature_extractor.extract_features(test_loader, split_name='test')
        
        extract_time = time.time() - extract_start
        logger.log(f"   Feature extraction time: {extract_time:.1f}s")
        
        # Save embeddings
        if config.SAVE_EMBEDDINGS:
            embeddings_data = {
                'train': {
                    'embeddings': train_embeddings,
                    'labels': train_labels,
                    'domains': train_domains,
                    'sections': train_sections,
                    'weights': train_data['weights']
                },
                'test': {
                    'embeddings': test_embeddings,
                    'labels': test_labels,
                    'domains': test_domains,
                    'sections': test_sections
                }
            }
            
            embeddings_file = os.path.join(config.EMBEDDINGS_DIR, f'{machine_id}_embeddings.pkl')
            with open(embeddings_file, 'wb') as f:
                pickle.dump(embeddings_data, f)
            logger.log(f"   💾 Embeddings saved: {embeddings_file}")
        
        # ============================================================
        # STEP 4: ANOMALY DETECTION (4-LAYER STRATEGY)
        # ============================================================
        detect_start = time.time()
        
        detector = FourLayerAnomalyDetector(config)
        detector.fit(
            train_embeddings, 
            train_domains, 
            train_sections,
            train_data['weights']
        )
        
        anomaly_scores = detector.predict(
            test_embeddings,
            test_domains,
            test_sections
        )
        
        detect_time = time.time() - detect_start
        logger.log(f"   Anomaly detection time: {detect_time:.1f}s")
        
        # ============================================================
        # STEP 5: COMPUTE METRICS
        # ============================================================
        logger.log(f"\n📊 Computing evaluation metrics...")
        metrics = compute_metrics(test_labels, anomaly_scores, test_domains)
        
        logger.log(f"\n{'='*80}")
        logger.log(f"RESULTS - {machine_id}")
        logger.log(f"{'='*80}")
        logger.log(f"  AUC (Overall):    {metrics['auc_overall']:>6.2f}%")
        logger.log(f"  pAUC (FPR≤0.1):   {metrics['pauc']:>6.2f}%")
        logger.log(f"  Official Score:   {metrics['official_score']:>6.2f}%")
        logger.log(f"  AUC (Source):     {metrics['auc_source']:>6.2f}%")
        logger.log(f"  AUC (Target):     {metrics['auc_target']:>6.2f}%")
        logger.log(f"{'='*80}")
        
        # ============================================================
        # STEP 6: VISUALIZE RESULTS
        # ============================================================
        logger.log(f"\n📈 Creating visualizations...")
        plot_results(metrics, machine_id, test_labels, anomaly_scores, 
                    test_domains, config)
        
        # ============================================================
        # STEP 7: SAVE DETAILED RESULTS
        # ============================================================
        logger.log(f"\n💾 Saving detailed results...")
        detailed_df = save_detailed_results(
            machine_id, test_data['files'], test_labels, 
            anomaly_scores, test_domains, test_sections, config
        )
        
        # ============================================================
        # STEP 8: COMPILE RESULTS
        # ============================================================
        machine_time = time.time() - machine_start_time
        
        results = {
            'machine': machine_id,
            'auc_overall': metrics['auc_overall'],
            'pauc': metrics['pauc'],
            'official_score': metrics['official_score'],
            'auc_source': metrics['auc_source'],
            'auc_target': metrics['auc_target'],
            'num_train': len(train_data['files']),
            'num_train_original': sum(train_data['weights'] == 1.0),
            'num_train_supplemental': sum(train_data['weights'] != 1.0),
            'num_test': len(test_data['files']),
            'num_anomalies': sum(test_labels),
            'num_normal': sum(test_labels == 0),
            'num_source': sum(test_domains == 0),
            'num_target': sum(test_domains == 1),
            'unique_sections': len(np.unique(test_sections)),
            'time_total': machine_time,
            'time_feature_extraction': extract_time,
            'time_anomaly_detection': detect_time,
            'used_supplemental': config.USE_SUPPLEMENTAL,
            'used_attributes': config.USE_ATTRIBUTES
        }
        
        # Save individual machine results
        results_file = os.path.join(config.RESULTS_DIR, f'{machine_id}_summary.json')
        with open(results_file, 'w') as f:
            json.dump(results, f, indent=2)
        
        logger.log(f"\n⏱️  Total time for {machine_id}: {machine_time/60:.2f} minutes")
        logger.log(f"   - Feature extraction: {extract_time:.1f}s ({extract_time/machine_time*100:.1f}%)")
        logger.log(f"   - Anomaly detection: {detect_time:.1f}s ({detect_time/machine_time*100:.1f}%)")
        
        # Update logger
        logger.update_machine_complete(machine_id, results)
        
        return results
        
    except Exception as e:
        logger.log(f"\n❌ ERROR processing {machine_id}:", level='error')
        logger.log(f"   {str(e)}", level='error')
        logger.log(f"\n{traceback.format_exc()}", level='error')
        logger.update_machine_failed(machine_id, str(e))
        
        return None

logger.log("✅ Main evaluation function loaded successfully!")


2026-01-20 06:38:04 | INFO | ✅ Main evaluation function loaded successfully!


In [11]:
# # ============================================================
# # MAIN EXECUTION - PROCESS ALL 7 MACHINES
# # ============================================================

# logger.log("\n" + "="*80)
# logger.log("🚀 STARTING EVALUATION OF ALL 7 MACHINES")
# logger.log("="*80)
# logger.log(f"Run ID: {RUN_TIMESTAMP}")
# logger.log(f"Configuration:")
# logger.log(f"  - Use Supplemental: {config.USE_SUPPLEMENTAL}")
# logger.log(f"  - Use Attributes: {config.USE_ATTRIBUTES}")
# logger.log(f"  - Multi-resolution spectrograms: {config.STFT_SIZES}")
# logger.log(f"  - 4-layer detection strategy enabled")
# logger.log("="*80)

# # Initialize PANNs feature extractor (shared across all machines)
# logger.log("\n🔧 Initializing PANNs feature extractor...")
# feature_extractor = PANNsFeatureExtractor(device=device)

# # Track overall progress
# experiment_start_time = time.time()
# all_results = []
# successful_machines = []
# failed_machines = []

# # Update progress tracker
# logger.progress['total_machines'] = len(config.MACHINES)
# logger.save_progress()

# # Process each machine
# for idx, machine_id in enumerate(config.MACHINES):
#     logger.log(f"\n{'#'*80}")
#     logger.log(f"# MACHINE {idx+1}/{len(config.MACHINES)}: {machine_id}")
#     logger.log(f"{'#'*80}")
    
#     try:
#         results = evaluate_machine(machine_id, config, feature_extractor)
        
#         if results is not None:
#             all_results.append(results)
#             successful_machines.append(machine_id)
#             logger.log(f"✅ Successfully processed {machine_id}")
#         else:
#             failed_machines.append(machine_id)
#             logger.log(f"⚠️  {machine_id} returned None", level='warning')
    
#     except KeyboardInterrupt:
#         logger.log("\n⚠️  Interrupted by user!", level='warning')
#         logger.log(f"Completed machines: {successful_machines}")
#         logger.log(f"Remaining machines: {config.MACHINES[idx+1:]}")
#         break
    
#     except Exception as e:
#         logger.log(f"❌ Unexpected error for {machine_id}: {e}", level='error')
#         failed_machines.append(machine_id)
#         continue
    
#     # Save progress after each machine
#     logger.save_progress()

# # ============================================================
# # COMPUTE FINAL SUMMARY
# # ============================================================

# total_time = time.time() - experiment_start_time

# logger.log("\n" + "="*80)
# logger.log("📊 FINAL SUMMARY - ALL MACHINES")
# logger.log("="*80)

# if len(all_results) > 0:
#     # Compute averages
#     avg_auc = np.mean([r['auc_overall'] for r in all_results])
#     avg_pauc = np.mean([r['pauc'] for r in all_results])
#     avg_official = np.mean([r['official_score'] for r in all_results])
#     avg_auc_source = np.mean([r['auc_source'] for r in all_results])
#     avg_auc_target = np.mean([r['auc_target'] for r in all_results])
    
#     # Print results table
#     logger.log(f"\n{'Machine':<15} {'AUC (%)':<10} {'pAUC (%)':<10} {'Official (%)':<12} {'AUC Src':<10} {'AUC Tgt':<10} {'Time (min)':<12}")
#     logger.log("-" * 100)
    
#     for r in all_results:
#         logger.log(f"{r['machine']:<15} {r['auc_overall']:>8.2f}  {r['pauc']:>8.2f}  "
#                   f"{r['official_score']:>10.2f}  {r['auc_source']:>8.2f}  "
#                   f"{r['auc_target']:>8.2f}  {r['time_total']/60:>10.2f}")
    
#     logger.log("-" * 100)
#     logger.log(f"{'AVERAGE':<15} {avg_auc:>8.2f}  {avg_pauc:>8.2f}  "
#               f"{avg_official:>10.2f}  {avg_auc_source:>8.2f}  "
#               f"{avg_auc_target:>8.2f}  {total_time/60/len(all_results):>10.2f}")
#     logger.log("=" * 100)
    
#     # Summary statistics
#     logger.log(f"\n📈 Performance Summary:")
#     logger.log(f"   Average AUC:           {avg_auc:.2f}%")
#     logger.log(f"   Average pAUC:          {avg_pauc:.2f}%")
#     logger.log(f"   Average Official Score: {avg_official:.2f}%")
#     logger.log(f"   AUC Range:             [{min(r['auc_overall'] for r in all_results):.2f}%, "
#               f"{max(r['auc_overall'] for r in all_results):.2f}%]")
    
#     logger.log(f"\n⏱️  Time Summary:")
#     logger.log(f"   Total time:            {total_time/60:.2f} minutes")
#     logger.log(f"   Average per machine:   {total_time/60/len(all_results):.2f} minutes")
#     logger.log(f"   Successful machines:   {len(successful_machines)}/{len(config.MACHINES)}")
    
#     if len(failed_machines) > 0:
#         logger.log(f"   Failed machines:       {len(failed_machines)}: {failed_machines}")
    
#     # Compile final summary
#     summary = {
#         'average_auc': float(avg_auc),
#         'average_pauc': float(avg_pauc),
#         'average_official_score': float(avg_official),
#         'average_auc_source': float(avg_auc_source),
#         'average_auc_target': float(avg_auc_target),
#         'min_auc': float(min(r['auc_overall'] for r in all_results)),
#         'max_auc': float(max(r['auc_overall'] for r in all_results)),
#         'total_time_minutes': total_time / 60,
#         'avg_time_per_machine_minutes': total_time / 60 / len(all_results),
#         'successful_machines': successful_machines,
#         'failed_machines': failed_machines,
#         'num_machines_processed': len(all_results),
#         'num_machines_total': len(config.MACHINES),
#         'per_machine_results': all_results,
#         'configuration': {
#             'use_supplemental': config.USE_SUPPLEMENTAL,
#             'use_attributes': config.USE_ATTRIBUTES,
#             'use_section_normalization': config.USE_SECTION_NORMALIZATION,
#             'use_mahalanobis': config.USE_MAHALANOBIS,
#             'stft_sizes': config.STFT_SIZES,
#             'k_values': config.K_VALUES
#         }
#     }
    
#     # Save final summary
#     summary_file = os.path.join(config.RESULTS_DIR, 'final_summary.json')
#     with open(summary_file, 'w') as f:
#         json.dump(summary, f, indent=2)
    
#     logger.log(f"\n💾 Final summary saved to: {summary_file}")
    
#     # Create comparison plot
#     if config.SAVE_PLOTS and len(all_results) > 1:
#         logger.log(f"\n📊 Creating comparison plot...")
        
#         fig, ax = plt.subplots(1, 1, figsize=(14, 8))
        
#         machines = [r['machine'].replace('dev_', '') for r in all_results]
#         aucs = [r['auc_overall'] for r in all_results]
#         colors = ['#2ecc71' if auc >= 65 else '#e74c3c' for auc in aucs]
        
#         bars = ax.bar(machines, aucs, color=colors, edgecolor='black', linewidth=1.5)
#         ax.axhline(y=avg_auc, color='blue', linestyle='--', linewidth=2, 
#                   label=f'Average: {avg_auc:.2f}%')
#         ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, 
#                   label='Random: 50%', alpha=0.5)
#         ax.axhline(y=65, color='orange', linestyle='--', linewidth=1, 
#                   label='Target: 65%', alpha=0.7)
        
#         ax.set_xlabel('Machine', fontsize=14, fontweight='bold')
#         ax.set_ylabel('AUC (%)', fontsize=14, fontweight='bold')
#         ax.set_title('DCASE 2025: AUC Comparison Across All Machines\n'
#                     f'PANNs CNN14 + Multi-Resolution + 4-Layer Detection', 
#                     fontsize=16, fontweight='bold')
#         ax.legend(fontsize=12, loc='lower right')
#         ax.grid(True, alpha=0.3, axis='y')
#         ax.set_ylim([0, 100])
        
#         # Add value labels on bars
#         for bar, auc in zip(bars, aucs):
#             height = bar.get_height()
#             ax.text(bar.get_x() + bar.get_width()/2., height + 1,
#                    f'{auc:.1f}%', ha='center', va='bottom', 
#                    fontsize=11, fontweight='bold')
        
#         plt.xticks(rotation=45, ha='right')
#         plt.tight_layout()
        
#         comparison_file = os.path.join(config.PLOTS_DIR, 'all_machines_comparison.png')
#         plt.savefig(comparison_file, dpi=150, bbox_inches='tight')
#         logger.log(f"   📊 Comparison plot saved: {comparison_file}")
#         plt.show()
#         plt.close()
    
#     # Finalize logger
#     logger.finalize(summary)
    
#     logger.log("\n" + "="*80)
#     logger.log("✅ EXPERIMENT COMPLETED SUCCESSFULLY!")
#     logger.log("="*80)
#     logger.log(f"📁 All results saved in: {RUN_DIR}")
#     logger.log(f"📊 Logs: {os.path.join(RUN_DIR, 'logs')}")
#     logger.log(f"📈 Plots: {os.path.join(RUN_DIR, 'plots')}")
#     logger.log(f"💾 Results: {os.path.join(RUN_DIR, 'results')}")
#     logger.log(f"🧠 Embeddings: {os.path.join(RUN_DIR, 'embeddings')}")
    
# else:
#     logger.log("❌ No machines were successfully processed!", level='error')
#     summary = {
#         'status': 'failed',
#         'failed_machines': failed_machines,
#         'total_time_minutes': total_time / 60
#     }
#     logger.finalize(summary)

# logger.log("\n🎉 Done! Check the run directory for all outputs.")


In [12]:
# ============================================================
# COMPLETE MINIMAL VERSION - ALL IN ONE
# ============================================================

# Update config
config.BASE_PATH = '/kaggle/input/dcase-dataset-2025/development/development'

# Simplified loader
def load_machine_data_minimal(machine_dir, machine_id, machine_name, config):
    """Load data with minimal logging"""
    machine_path = os.path.join(machine_dir, machine_id, machine_name)
    train_dir = os.path.join(machine_path, 'train')
    test_dir = os.path.join(machine_path, 'test')
    supplemental_dir = os.path.join(machine_path, 'supplemental')
    attributes_file = os.path.join(machine_path, 'attributes_00.csv')
    
    # Load attributes silently
    attributes_df = None
    if config.USE_ATTRIBUTES and os.path.exists(attributes_file):
        try:
            attributes_df = pd.read_csv(attributes_file)
        except:
            pass
    
    # Training data
    train_files = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith('.wav')])
    train_labels, train_domains, train_sections, train_weights = [], [], [], []
    
    for f in train_files:
        section, domain, label = parse_filename(f)
        train_labels.append(label)
        train_domains.append(domain)
        train_sections.append(section)
        train_weights.append(1.0)
    
    # Supplemental data
    supplemental_files = []
    supplemental_labels, supplemental_domains, supplemental_sections, supplemental_weights = [], [], [], []
    
    if config.USE_SUPPLEMENTAL and os.path.exists(supplemental_dir):
        all_supplemental = sorted([os.path.join(supplemental_dir, f) 
                                  for f in os.listdir(supplemental_dir) if f.endswith('.wav')])
        
        if len(all_supplemental) > config.MAX_SUPPLEMENTAL_SAMPLES:
            source_supp = [f for f in all_supplemental if 'source' in f]
            target_supp = [f for f in all_supplemental if 'target' in f]
            n_source = min(len(source_supp), config.MAX_SUPPLEMENTAL_SAMPLES // 2)
            n_target = min(len(target_supp), config.MAX_SUPPLEMENTAL_SAMPLES // 2)
            np.random.seed(42)
            supplemental_files = (list(np.random.choice(source_supp, n_source, replace=False)) +
                                 list(np.random.choice(target_supp, n_target, replace=False)))
        else:
            supplemental_files = all_supplemental
        
        for f in supplemental_files:
            section, domain, label = parse_filename(f)
            supplemental_labels.append(0)
            supplemental_domains.append(domain)
            supplemental_sections.append(section)
            supplemental_weights.append(config.SUPPLEMENTAL_WEIGHT)
    
    # Combine
    all_train_files = train_files + supplemental_files
    all_train_labels = train_labels + supplemental_labels
    all_train_domains = train_domains + supplemental_domains
    all_train_sections = train_sections + supplemental_sections
    all_train_weights = train_weights + supplemental_weights
    
    # Test data
    test_files = sorted([os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.wav')])
    test_labels, test_domains, test_sections = [], [], []
    
    for f in test_files:
        section, domain, label = parse_filename(f)
        test_labels.append(label)
        test_domains.append(domain)
        test_sections.append(section)
    
    train_data = {
        'files': all_train_files,
        'labels': np.array(all_train_labels),
        'domains': np.array(all_train_domains),
        'sections': np.array(all_train_sections),
        'weights': np.array(all_train_weights)
    }
    
    test_data = {
        'files': test_files,
        'labels': np.array(test_labels),
        'domains': np.array(test_domains),
        'sections': np.array(test_sections)
    }
    
    return train_data, test_data, attributes_df

# Simplified evaluation
def evaluate_machine_minimal(machine_id, config, feature_extractor):
    """Minimal version - only essential output"""
    machine_start = time.time()
    
    try:
        machine_name = config.MACHINE_NAMES[machine_id]
        print(f"🔄 {machine_id}...", end=' ', flush=True)
        
        # Load data
        train_data, test_data, _ = load_machine_data_minimal(
            config.BASE_PATH, machine_id, machine_name, config
        )
        
        # Create datasets
        train_dataset = DcaseDataset(train_data['files'], train_data['labels'],
                                    train_data['domains'], train_data['sections'], config)
        test_dataset = DcaseDataset(test_data['files'], test_data['labels'],
                                   test_data['domains'], test_data['sections'], config)
        
        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                                 shuffle=False, num_workers=0, pin_memory=False)
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE,
                                shuffle=False, num_workers=0, pin_memory=False)
        
        # Extract features (suppress tqdm output)
        from tqdm.auto import tqdm
        original_disable = tqdm.__init__.__defaults__
        tqdm.__init__.__defaults__ = (None,) * 9 + (True,) + original_disable[10:]
        
        train_embeddings, train_labels, train_domains, train_sections = \
            feature_extractor.extract_features(train_loader, split_name='train')
        test_embeddings, test_labels, test_domains, test_sections = \
            feature_extractor.extract_features(test_loader, split_name='test')
        
        # Save embeddings
        if config.SAVE_EMBEDDINGS:
            with open(os.path.join(config.EMBEDDINGS_DIR, f'{machine_id}_embeddings.pkl'), 'wb') as f:
                pickle.dump({
                    'train': {'embeddings': train_embeddings, 'labels': train_labels,
                             'domains': train_domains, 'sections': train_sections, 
                             'weights': train_data['weights']},
                    'test': {'embeddings': test_embeddings, 'labels': test_labels,
                            'domains': test_domains, 'sections': test_sections}
                }, f)
        
        # Anomaly detection
        detector = FourLayerAnomalyDetector(config)
        detector.fit(train_embeddings, train_domains, train_sections, train_data['weights'])
        anomaly_scores = detector.predict(test_embeddings, test_domains, test_sections)
        
        # Compute metrics
        metrics = compute_metrics(test_labels, anomaly_scores, test_domains)
        machine_time = time.time() - machine_start
        
        results = {
            'machine': machine_id,
            'auc_overall': metrics['auc_overall'],
            'pauc': metrics['pauc'],
            'official_score': metrics['official_score'],
            'auc_source': metrics['auc_source'],
            'auc_target': metrics['auc_target'],
            'num_train': len(train_data['files']),
            'num_test': len(test_data['files']),
            'time_minutes': machine_time / 60
        }
        
        # Save results
        with open(os.path.join(config.RESULTS_DIR, f'{machine_id}_results.json'), 'w') as f:
            json.dump(results, f, indent=2)
        
        if config.SAVE_DETAILED_RESULTS:
            pd.DataFrame({
                'filename': [os.path.basename(f) for f in test_data['files']],
                'label': test_labels,
                'domain': test_domains,
                'section': test_sections,
                'anomaly_score': anomaly_scores
            }).to_csv(os.path.join(config.RESULTS_DIR, f'{machine_id}_predictions.csv'), index=False)
        
        print(f"✅ AUC: {results['auc_overall']:.2f}% ({machine_time/60:.1f}min)")
        return results
        
    except Exception as e:
        print(f"❌ Error: {str(e)[:50]}")
        return None

print("✅ Minimal version ready!")


✅ Minimal version ready!


In [13]:
# ============================================================
# SIMPLIFIED: Raw Audio Only (Better for PANNs!)
# ============================================================

print("✅ Switching to raw audio input (better for PANNs!)")

# Simple dataset - just raw audio
class SimpleAudioDataset(Dataset):
    """Dataset that loads raw audio only"""
    
    def __init__(self, file_paths, labels, domains, sections, config):
        self.file_paths = file_paths
        self.labels = labels
        self.domains = domains
        self.sections = sections
        self.config = config
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        # Load raw audio
        audio, _ = librosa.load(self.file_paths[idx], sr=self.config.SAMPLE_RATE, mono=True)
        
        # Pad or trim to 10 seconds
        if len(audio) < self.config.AUDIO_LEN:
            audio = np.pad(audio, (0, self.config.AUDIO_LEN - len(audio)))
        else:
            audio = audio[:self.config.AUDIO_LEN]
        
        return {
            'audio': torch.FloatTensor(audio),
            'label': self.labels[idx],
            'domain': self.domains[idx],
            'section': self.sections[idx]
        }

# Updated minimal evaluation
def evaluate_machine_raw_audio(machine_id, config, feature_extractor):
    """Evaluation using raw audio (no spectrograms)"""
    machine_start = time.time()
    
    try:
        machine_name = config.MACHINE_NAMES[machine_id]
        print(f"🔄 {machine_id}...", end=' ', flush=True)
        
        # Load data
        train_data, test_data, _ = load_machine_data_minimal(
            config.BASE_PATH, machine_id, machine_name, config
        )
        
        # Create datasets with raw audio only
        train_dataset = SimpleAudioDataset(train_data['files'], train_data['labels'],
                                          train_data['domains'], train_data['sections'], config)
        test_dataset = SimpleAudioDataset(test_data['files'], test_data['labels'],
                                         test_data['domains'], test_data['sections'], config)
        
        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                                 shuffle=False, num_workers=0)
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE,
                                shuffle=False, num_workers=0)
        
        # Extract features from raw audio
        train_embeddings, train_labels, train_domains, train_sections = \
            feature_extractor.extract_features(train_loader, split_name='train')
        test_embeddings, test_labels, test_domains, test_sections = \
            feature_extractor.extract_features(test_loader, split_name='test')
        
        # Save embeddings
        if config.SAVE_EMBEDDINGS:
            with open(os.path.join(config.EMBEDDINGS_DIR, f'{machine_id}_embeddings.pkl'), 'wb') as f:
                pickle.dump({
                    'train': {'embeddings': train_embeddings, 'labels': train_labels,
                             'domains': train_domains, 'sections': train_sections, 
                             'weights': train_data['weights']},
                    'test': {'embeddings': test_embeddings, 'labels': test_labels,
                            'domains': test_domains, 'sections': test_sections}
                }, f)
        
        # 4-layer anomaly detection
        detector = FourLayerAnomalyDetector(config)
        detector.fit(train_embeddings, train_domains, train_sections, train_data['weights'])
        anomaly_scores = detector.predict(test_embeddings, test_domains, test_sections)
        
        # Compute metrics
        metrics = compute_metrics(test_labels, anomaly_scores, test_domains)
        machine_time = time.time() - machine_start
        
        results = {
            'machine': machine_id,
            'auc_overall': metrics['auc_overall'],
            'pauc': metrics['pauc'],
            'official_score': metrics['official_score'],
            'auc_source': metrics['auc_source'],
            'auc_target': metrics['auc_target'],
            'num_train': len(train_data['files']),
            'num_test': len(test_data['files']),
            'time_minutes': machine_time / 60
        }
        
        # Save results
        with open(os.path.join(config.RESULTS_DIR, f'{machine_id}_results.json'), 'w') as f:
            json.dump(results, f, indent=2)
        
        if config.SAVE_DETAILED_RESULTS:
            pd.DataFrame({
                'filename': [os.path.basename(f) for f in test_data['files']],
                'label': test_labels,
                'domain': test_domains,
                'section': test_sections,
                'anomaly_score': anomaly_scores
            }).to_csv(os.path.join(config.RESULTS_DIR, f'{machine_id}_predictions.csv'), index=False)
        
        print(f"✅ AUC: {results['auc_overall']:.2f}% ({machine_time/60:.1f}min)")
        return results
        
    except Exception as e:
        print(f"❌ {str(e)[:50]}")
        return None

print("✅ Raw audio version ready! Run Cell 16 to execute.")


✅ Switching to raw audio input (better for PANNs!)
✅ Raw audio version ready! Run Cell 16 to execute.


In [14]:
# ============================================================
# MAIN EXECUTION - RAW AUDIO VERSION
# ============================================================

print("="*80)
print(f"🚀 DCASE 2025 - PANNs (Raw Audio) + 4-Layer Detection")
print(f"📁 Run: {RUN_TIMESTAMP}")
print("="*80)

# Initialize PANNs (only once)
print("\n🔧 Loading PANNs CNN14...", end=' ', flush=True)
feature_extractor = PANNsFeatureExtractor(device=device)
print("✅")

# Process all machines
start_time = time.time()
all_results = []

print(f"\n📊 Processing {len(config.MACHINES)} machines:\n")

for idx, machine_id in enumerate(config.MACHINES, 1):
    print(f"[{idx}/{len(config.MACHINES)}] ", end='')
    result = evaluate_machine_raw_audio(machine_id, config, feature_extractor)  # ← CHANGED THIS
    if result:
        all_results.append(result)

# Final summary
total_time = time.time() - start_time

print("\n" + "="*80)
print("📊 FINAL RESULTS")
print("="*80)

if len(all_results) > 0:
    # Compute averages
    avg_auc = np.mean([r['auc_overall'] for r in all_results])
    avg_pauc = np.mean([r['pauc'] for r in all_results])
    avg_official = np.mean([r['official_score'] for r in all_results])
    
    # Print table
    print(f"\n{'Machine':<15} {'AUC%':>8} {'pAUC%':>8} {'Official%':>10} {'Time(min)':>10}")
    print("-"*60)
    for r in all_results:
        print(f"{r['machine']:<15} {r['auc_overall']:>8.2f} {r['pauc']:>8.2f} "
              f"{r['official_score']:>10.2f} {r['time_minutes']:>10.1f}")
    print("-"*60)
    print(f"{'AVERAGE':<15} {avg_auc:>8.2f} {avg_pauc:>8.2f} {avg_official:>10.2f} "
          f"{total_time/60:>10.1f}")
    print("="*60)
    
    # Save final summary
    summary = {
        'average_auc': float(avg_auc),
        'average_pauc': float(avg_pauc),
        'average_official_score': float(avg_official),
        'total_time_minutes': total_time / 60,
        'timestamp': RUN_TIMESTAMP,
        'per_machine_results': all_results
    }
    
    with open(os.path.join(config.RESULTS_DIR, 'final_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n✅ Results saved to: {config.RESULTS_DIR}")
    print(f"📊 Summary: final_summary.json")
    print(f"💾 Embeddings: {config.EMBEDDINGS_DIR}")
    
else:
    print("❌ No machines processed successfully")

print(f"\n⏱️  Total time: {total_time/60:.1f} minutes")
print("="*80)


🚀 DCASE 2025 - PANNs (Raw Audio) + 4-Layer Detection
📁 Run: 20260120_063740

🔧 Loading PANNs CNN14... 2026-01-20 06:38:04 | INFO | 
🔧 Initializing PANNs CNN14 model...
Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth


--2026-01-20 06:38:04--  http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.132.207, 142.250.152.207, 142.251.184.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.132.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14675 (14K) [application/octet-stream]
Saving to: ‘/root/panns_data/class_labels_indices.csv’

     0K .......... ....                                       100%  134M=0s

2026-01-20 06:38:04 (134 MB/s) - ‘/root/panns_data/class_labels_indices.csv’ saved [14675/14675]

--2026-01-20 06:38:04--  https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 137.138.52.235, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/3987831/files

Using CPU.
2026-01-20 06:41:14 | INFO | ✅ PANNs CNN14 loaded successfully!
2026-01-20 06:41:14 | INFO |    Model device: cuda
2026-01-20 06:41:14 | INFO |    Pre-trained on AudioSet (2M audio clips)
2026-01-20 06:41:14 | INFO |    Output embedding dimension: 2048
✅

📊 Processing 7 machines:

[1/7] 🔄 dev_fan... 2026-01-20 06:41:14 | INFO | 
🎵 Extracting features for train split...


Extracting train features:   0%|          | 0/69 [00:00<?, ?it/s]

2026-01-20 06:47:12 | INFO | ✅ Feature extraction complete for train
2026-01-20 06:47:12 | INFO |    Embeddings shape: (1100, 2048)
2026-01-20 06:47:12 | INFO |    Time: 357.0s (3.1 samples/s)
2026-01-20 06:47:12 | INFO |    Mean embedding norm: 11.375
2026-01-20 06:47:12 | INFO |    Std embedding norm: 1.022
2026-01-20 06:47:12 | INFO | 
🎵 Extracting features for test split...


...

2026-01-20 07:18:32 | INFO | ✅ Feature extraction complete for test
2026-01-20 07:18:32 | INFO |    Embeddings shape: (200, 2048)
2026-01-20 07:18:32 | INFO |    Time: 57.3s (3.5 samples/s)
2026-01-20 07:18:32 | INFO |    Mean embedding norm: 10.667
2026-01-20 07:18:32 | INFO |    Std embedding norm: 0.282
2026-01-20 07:18:32 | INFO | 
🎯 Fitting anomaly detector...
2026-01-20 07:18:32 | INFO |    Training samples: 1100
2026-01-20 07:18:32 | INFO |    Embedding dimension: 2048
2026-01-20 07:18:32 | INFO |    Unique sections: 1
2026-01-20 07:18:33 | INFO |    Domain 0 baseline: μ=1.2023, σ=0.1777, n=990
2026-01-20 07:18:33 | INFO |    Domain 1 baseline: μ=0.9037, σ=0.2633, n=110
2026-01-20 07:18:33 | INFO |    Section-specific stats computed for 1 sections
2026-01-20 07:18:33 | INFO | 
🔮 Computing anomaly scores...
2026-01-20 07:18:33 | INFO |    Test samples: 200
2026-01-20 07:18:33 | INFO |    Layer 1 (Multi-k): 0.11s
2026-01-20 07:18:34 | INFO |    Layer 2 (Mahalanobis): 1.12s
2026-01

Extracting train features:   0%|          | 0/69 [00:00<?, ?it/s]

2026-01-20 07:23:47 | INFO | ✅ Feature extraction complete for train
2026-01-20 07:23:47 | INFO |    Embeddings shape: (1100, 2048)
2026-01-20 07:23:47 | INFO |    Time: 312.9s (3.5 samples/s)
2026-01-20 07:23:47 | INFO |    Mean embedding norm: 11.426
2026-01-20 07:23:47 | INFO |    Std embedding norm: 0.665
2026-01-20 07:23:47 | INFO | 
🎵 Extracting features for test split...


Extracting test features:   0%|          | 0/13 [00:00<?, ?it/s]

2026-01-20 07:24:42 | INFO | ✅ Feature extraction complete for test
2026-01-20 07:24:42 | INFO |    Embeddings shape: (200, 2048)
2026-01-20 07:24:42 | INFO |    Time: 54.9s (3.6 samples/s)
2026-01-20 07:24:42 | INFO |    Mean embedding norm: 11.300
2026-01-20 07:24:42 | INFO |    Std embedding norm: 0.705
2026-01-20 07:24:42 | INFO | 
🎯 Fitting anomaly detector...
2026-01-20 07:24:42 | INFO |    Training samples: 1100
2026-01-20 07:24:42 | INFO |    Embedding dimension: 2048
2026-01-20 07:24:42 | INFO |    Unique sections: 1
2026-01-20 07:24:42 | INFO |    Domain 0 baseline: μ=0.8100, σ=0.1687, n=1090
2026-01-20 07:24:42 | INFO |    Domain 1 baseline: μ=0.8580, σ=0.1639, n=10
2026-01-20 07:24:42 | INFO |    Section-specific stats computed for 1 sections
2026-01-20 07:24:42 | INFO | 
🔮 Computing anomaly scores...
2026-01-20 07:24:42 | INFO |    Test samples: 200
2026-01-20 07:24:43 | INFO |    Layer 1 (Multi-k): 0.10s
2026-01-20 07:24:43 | INFO |    Layer 2 (Mahalanobis): 0.92s
2026-01